# Clase 07 · Tratamiento de outliers — práctica guiada
Seguimos el notebook del profesor (dataset Boston), pero sobre **`data/raw/viviendas_lima.csv`**.

1. Impacto de un outlier en media, mediana, moda y desviación estándar
2. Detección univariada: IQR, percentiles 5–95, ±3 desviaciones estándar
3. Eliminar con IQR una columna (y por qué "siguen apareciendo" outliers)
4. Eliminar en todas las columnas a la vez: cuántas filas se pierden y qué se sesga
5. Outliers multivariados: altura/peso, distancia de Mahalanobis
6. 📌 Extra: IsolationForest y LocalOutlierFactor de scikit-learn (lo que PyOD empaqueta)

> Las celdas de gráficos son **opcionales**: si no tienes matplotlib instalado, simplemente se saltan.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Busca la raíz del repo (la carpeta que contiene data/) subiendo desde aquí
raiz = Path.cwd()
while not (raiz / "data" / "generar_datos.py").exists() and raiz != raiz.parent:
    raiz = raiz.parent
ruta = raiz / "data" / "raw" / "viviendas_lima.csv"
if not ruta.exists():
    import sys; sys.path.insert(0, str(raiz))
    from data.generar_datos import main as generar
    generar()
df = pd.read_csv(ruta)
print(df.shape)
df.head()

In [ ]:
# Gráficos opcionales: si no hay matplotlib, HAY_PLT = False y esas celdas no dibujan nada
try:
    import matplotlib.pyplot as plt
    HAY_PLT = True
except ImportError:
    HAY_PLT = False
    print("matplotlib no está instalado: se omiten los gráficos (el resto funciona igual).")

## 1. Un solo outlier "malogra" la distribución
El profesor: la media sube de ~5 a ~30 al agregar un 300; la mediana casi no cambia, la moda sigue igual y la desviación estándar se dispara porque depende de la media.

In [ ]:
lista = pd.Series([2, 3, 4, 5, 5, 5, 6, 7, 7, 8, 8])
con_300 = pd.concat([lista, pd.Series([300])], ignore_index=True)

resumen = pd.DataFrame({
    "sin outlier": [lista.mean(), lista.median(), lista.mode()[0], lista.std()],
    "con 300": [con_300.mean(), con_300.median(), con_300.mode()[0], con_300.std()],
}, index=["media", "mediana", "moda", "desv. estándar"]).round(2)
resumen

## 2. Detección univariada
### 2.1 Regla IQR
`IQR = Q3 − Q1`, límites `Q1 − 1.5·IQR` y `Q3 + 1.5·IQR`. Es "la más clásica" porque no tiene hiperparámetros.

In [ ]:
s = df["area_m2"]
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}")
print(f"límites: [{inferior:.2f}, {superior:.2f}]")

es_outlier = (s < inferior) | (s > superior)
print("outliers por debajo:", int((s < inferior).sum()), "| por encima:", int((s > superior).sum()))

¿Quiénes son esos 147 "outliers"? Igual que con `CRIM`, por abajo no cae nadie. Pero mira **qué tipo de inmueble** son:

In [ ]:
print(df.loc[es_outlier, "tipo_inmueble"].value_counts())
print()
print("Áreas > 1000 m² (errores ×10 evidentes):", int((s > 1000).sum()))
df.loc[es_outlier].sort_values("area_m2", ascending=False)[["tipo_inmueble", "area_m2", "precio_usd"]].head(8)

La mayoría son **casas grandes reales**, no errores. La regla IQR no sabe distinguir "error" de "caso poco común".

In [ ]:
if HAY_PLT:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].boxplot(df["area_m2"].dropna(), vert=False); ax[0].set_title("area_m2 (todo)")
    ax[1].boxplot(df.loc[df["area_m2"] < 1000, "area_m2"], vert=False); ax[1].set_title("area_m2 < 1000")
    plt.tight_layout(); plt.show()

### 2.2 Percentiles 5–95 y ±3 desviaciones estándar
Las otras dos reglas que mencionó el profesor, comparadas en `precio_usd`:

In [ ]:
p = df["precio_usd"]

# IQR
q1, q3 = p.quantile([0.25, 0.75]); iqr = q3 - q1
n_iqr = int(((p < q1 - 1.5 * iqr) | (p > q3 + 1.5 * iqr)).sum())

# Percentiles 5 y 95: SIEMPRE deja fuera ~10 % de los datos
p5, p95 = p.quantile([0.05, 0.95])
n_pct = int(((p < p5) | (p > p95)).sum())

# 3 desviaciones estándar
z = (p - p.mean()) / p.std()
n_z = int((z.abs() > 3).sum())

pd.Series({"IQR (1.5)": n_iqr, "percentiles 5-95": n_pct, "|z| > 3": n_z}, name="filas marcadas")

📌 La regla de 3σ marca **menos**: las mansiones (precio ×5–9) inflan la media y la desviación estándar, y así "se esconden" un poco.

## 3. Eliminar con IQR en una columna... y volver a mirar
El profesor eliminó los outliers de `CRIM` y *"todavía hay muchos outliers"*: los cuartiles se recalculan con los datos nuevos.

In [ ]:
def fuera_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75]); iqr = q3 - q1
    return (serie < q1 - k * iqr) | (serie > q3 + k * iqr)

datos = df.copy()
for pasada in range(1, 5):
    marcados = fuera_iqr(datos["area_m2"])
    print(f"pasada {pasada}: {len(datos)} filas, {int(marcados.sum())} outliers, máx = {datos['area_m2'].max():.1f}")
    datos = datos[~marcados]

Cada vez que borras, la "caja" se achica y aparecen outliers nuevos. Borrar en bucle no tiene fin: **no es una buena estrategia**.

## 4. Todas las columnas a la vez (la "automatización")
Así lo hizo el profesor: percentiles 25 y 75 de todas las columnas y un solo filtro. En Boston pasó de 506 a 268 filas.

In [ ]:
cols = ["area_m2", "precio_usd", "antiguedad_anios", "distancia_centro_km"]
q1 = df[cols].quantile(0.25)
q3 = df[cols].quantile(0.75)
iqr = q3 - q1
fuera = (df[cols] < q1 - 1.5 * iqr) | (df[cols] > q3 + 1.5 * iqr)   # NaN -> False (no es outlier)
df_limpio = df[~fuera.any(axis=1)].copy()

perdido = (len(df) - len(df_limpio)) / len(df) * 100
print(f"{len(df)} -> {len(df_limpio)} filas  (se pierde {perdido:.2f} %)")
print("outliers por columna:"); print(fuera.sum())

### ¿Qué se sesgó?
El profesor: *"me puede comprometer las otras variables... la distribución me la puede sesgar"*. Mira `tipo_inmueble`, que **ni siquiera estaba** en la lista de columnas:

In [ ]:
comparacion = pd.DataFrame({
    "original": df["tipo_inmueble"].value_counts(normalize=True),
    "tras borrar outliers": df_limpio["tipo_inmueble"].value_counts(normalize=True),
}).round(3)
comparacion

Las casas bajan de ~22.5 % a ~15.2 %. Un modelo de precios entrenado con `df_limpio` vería muchas menos casas de las que hay en el mercado.

### ✏️ Tu turno 1
Agrega `"piso"` a `cols` y repite la eliminación. ¿Cuántas filas quedan ahora? ¿Qué pasa con la proporción de departamentos (los que tienen piso alto)?

In [ ]:
# Tu código aquí


## 5. Outliers multivariados: altura y peso
Ninguna columna tiene outliers por separado, pero p15 (182 cm, 52 kg) es rara en conjunto.

In [ ]:
ap = pd.DataFrame({
    "altura_cm": [150, 155, 158, 160, 163, 165, 168, 170, 173, 175, 178, 180, 183, 185, 182],
    "peso_kg":   [ 50,  53,  55,  57,  60,  62,  64,  66,  69,  71,  74,  76,  79,  82,  52],
}, index=[f"p{i:02d}" for i in range(1, 16)])

print("outliers IQR en altura:", int(fuera_iqr(ap["altura_cm"]).sum()))
print("outliers IQR en peso:  ", int(fuera_iqr(ap["peso_kg"]).sum()))

In [ ]:
if HAY_PLT:
    plt.figure(figsize=(4, 3))
    plt.scatter(ap["altura_cm"], ap["peso_kg"])
    plt.scatter(ap.loc["p15", "altura_cm"], ap.loc["p15", "peso_kg"], color="red", label="p15")
    plt.xlabel("altura_cm"); plt.ylabel("peso_kg"); plt.legend(); plt.show()

### 📌 Extra: distancia de Mahalanobis
Mide la distancia al centro teniendo en cuenta la **covarianza**: alejarse a lo largo de la diagonal (alto y pesado) es normal; salirse de ella (alto y liviano) no.

Primero veamos que la distancia "normal" (euclidiana) **no** ve a p15 como especial. Luego usamos la versión de scikit-learn. En el ejercicio (función 6) la implementarás tú con numpy, paso a paso.

In [ ]:
X = ap.to_numpy(dtype=float)
centro = X.mean(axis=0)
print("covarianza:")
print(np.cov(X, rowvar=False).round(1))

# Distancia euclidiana "normal" al centro
euclid = pd.Series(np.sqrt(((X - centro) ** 2).sum(axis=1)), index=ap.index)
print()
print("Distancia euclidiana (top 3):")
print(euclid.nlargest(3).round(2))

In [ ]:
# Mahalanobis con scikit-learn (EmpiricalCovariance devuelve la distancia AL CUADRADO)
from sklearn.covariance import EmpiricalCovariance

ec = EmpiricalCovariance().fit(X)          # ojo: usa ddof=0, valores un poco distintos a np.cov (ddof=1)
maha = pd.Series(np.sqrt(ec.mahalanobis(X)), index=ap.index)
print("Mahalanobis (top 3):")
print(maha.nlargest(3).round(2))

Con la distancia euclidiana ganan las personas de los extremos (muy bajas o muy altas). Con Mahalanobis, **p15** destaca claramente.

### ✏️ Tu turno 2
Cambia p15 a `(182, 78)` (alto y pesado, coherente con la diagonal). ¿Sigue siendo la mayor distancia de Mahalanobis? ¿Por qué?

In [ ]:
# Tu código aquí


## 6. 📌 Extra: lo que empaqueta PyOD (versión scikit-learn)
El transcript se corta cuando el profesor presenta **PyOD**. Dos de sus algoritmos más usados vienen también en scikit-learn:
- **IsolationForest**: los outliers se aíslan con pocos cortes aleatorios.
- **LocalOutlierFactor (LOF)**: un punto en una zona mucho menos densa que sus vecinos es raro.

Ambos devuelven `-1` = outlier, `1` = normal. Los aplicamos a (`area_m2`, `precio_usd`):

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

Xv = df[["area_m2", "precio_usd"]].dropna()

iso = IsolationForest(contamination=0.01, random_state=42)
etiqueta_iso = pd.Series(iso.fit_predict(Xv), index=Xv.index)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
etiqueta_lof = pd.Series(lof.fit_predict(Xv), index=Xv.index)

print("IsolationForest marca:", int((etiqueta_iso == -1).sum()), "filas")
print("LOF marca:            ", int((etiqueta_lof == -1).sum()), "filas")
print("Marcadas por ambos:   ", int(((etiqueta_iso == -1) & (etiqueta_lof == -1)).sum()))
df.loc[etiqueta_iso[etiqueta_iso == -1].index, ["tipo_inmueble", "area_m2", "precio_usd"]].sort_values("area_m2").tail(10)

⚠️ `contamination` **es un hiperparámetro**: le dices al algoritmo qué proporción de outliers esperas. Justo lo que el profesor destacaba que la regla IQR **no** necesita.

📌 Además, `area_m2` y `precio_usd` tienen escalas muy distintas (decenas vs cientos de miles). Para LOF (que usa distancias) conviene escalar antes; Mahalanobis ya corrige la escala por sí sola.

### ✏️ Tu turno 3
1. Escala `Xv` con `sklearn.preprocessing.StandardScaler` antes de LOF. ¿Cambian las filas marcadas?
2. ¿Cuántas filas con `area_m2 > 1000` o `precio_usd > 1_000_000` detecta IsolationForest? Prueba también `contamination=0.02`.

In [ ]:
# Tu código aquí


## Resumen
| Paso | Lo que viste |
|---|---|
| Impacto | Un 300 lleva la media de 5.45 a 30; la mediana casi no se mueve |
| IQR en una columna | 147 "outliers" en `area_m2`, la mayoría casas reales |
| Repetir IQR | Siempre aparecen outliers nuevos |
| IQR en todas | 12.5 % de filas perdidas y casas subrepresentadas (sesgo) |
| Multivariado | p15 invisible por columna, evidente con Mahalanobis |

Ahora ve a **`ejercicio.py`** y evalúate con `python -m framework evaluar <ruta-de-esta-carpeta>`.